# Streams

Welcome to the documentation the *Streams* part of *Kafi Streams*.

Streams is stream processing made natural.

## Your First Streams Topology

In Streams, you create a *processing topology* or just *topology* with an arbitrary number of *sources* (=inputs) and an arbitrary number of *sinks* (=outputs).

Typically, sources and sinks are Kafka topics.

So let's do something.

Our source is a Kafka topic of orders like this:
```json
{"key": 42,
 "value": {"id": 42,
           "product_id": 4711,
           "customer_id": 23,
           "ts": 1609459200000},
 "partition": 3,
 "offset": 47,
 "timestamp": [1, 1784196912929],
 "headers": null}
```

Our first humble aim is to group these orders by customer (`customer_id`) and aggregate the respective orders and products.

How does this look like in Streams?


In [ ]:
import sys
sys.path.insert(1, "..")

import kafi.streams.streams
import importlib
importlib.reload(kafi.streams.streams)

from kafi.fs.local.local import Local
from kafi.streams.streams import Streams, start_streams_task

# "Connect" to local Kafka emulation
c = Local({"local": {"root.dir": "/tmp"}})

# Source + sink topic names
source_str = "orders"
sink_str = "orders_aggregated"

# Create the Streams topology
tn = (
    # 1. Specify the source.
    Streams.source(c, source_str).peek("source")
    # 2. We are only interested in the value of the messages.
    .map(lambda r: r["value"]).peek("map1")
    # 3. The group by/aggregation.
    .group_by_agg(lambda r: r["customer_id"],
                  lambda r: r,
                  lambda agg_r, r: {"orders": agg_r["orders"] + 1,
                                    "product_ids": agg_r["product_ids"] + [r["product_id"]]},
                  {"orders": 0, "product_ids": []},
                  lambda by, agg_r: {"customer_id": by,
                                     "orders": agg_r["orders"],
                                     "product_ids": agg_r["product_ids"]}).peek("group_by_agg")
    # 4. Recreate the Kafka message for the sink.
    .map(lambda r: {"key": r["customer_id"],
                    "value": r}).peek("map2")
    # 6. Specify the sink.
    .sink(c, sink_str)
)


## Setup

For simplicity, we use Kafi's emulated Kafka on your local disk (replace the `root.dir` with your preferred directory where you'd like Kafi's emulated Kafka to put its files).

The interesting part is the topology specification `tn=...`. It consists of five steps (excluding the `peek` calls for debugging, you can ignore them for the time being):

1. We specify the source. Unlike Kafka Streams, each source can be on any Kafka cluster (here: `c`). The source topic name is `orders` (=`source_str`)
2. We are only interested in the `value` of the messages - hence, we just select that from the Kafka messages. In principle, you can access any field from the Kafka messages, not just the key and value but also the partition, offset, timestamp and the headers.
3. We group the messages by their `customer_id` and aggregate the respective orders and products. The "projection" of the aggregation is a record including the `customer_id`, `orders` and `product_ids`:
    * orders are just counted in `orders`
    * the product IDs are accumulated in the list `product_ids`
4. We recreate the Kafka message layout - putting the `customer_id` into the key and the aggregated record into the value.
5. We specify the sink. Again, unlike Kafka Streams, each sink can be on any Kafka cluster (here: `c` again). The sink topic name is `orders_aggregated` (=`sink_str`)

That's it. Ready to rumble.

Let's first create a generator for randomly generating orders:

In [ ]:
import random

class OrderGenerator:
    def __init__(self):
        self.order_id_int = 0
        self.customer_id_int = 0
        #
        self.ts_int = 0
        self.ts_step_int = 1

    def generate(self):
        message_dict = {
            "key": self.order_id_int,
            "value": {"id": self.order_id_int,
                      "product_id": random.randint(0, 100 - 1),
                      "customer_id": random.randint(0, 10 - 1),
                      "ts": self.ts_int},
        }
        #
        self.order_id_int += 1
        #
        self.ts_int += self.ts_step_int
        #
        return message_dict

#

gen = OrderGenerator()
for _ in range(3):
    print(gen.generate())


And then, in the next step:
1. "Build" the topology specified above to make it ready for running.
2. (Re-)create the source and sink topics.
3. Start Streams as a Python task.

...and:

4. Get the generator.
5. Set up the producer.
6. 10 times: Generate a message and produce it to Kafka.
7. Close the producer.

Since we added `peek` calls after each step, you'll see the outputs of the individual steps now as Streams picks up the produced messages.

In [ ]:
# 1. Build the topology.
built_tn = Streams.build(tn)
# 2. (Re-)create the source and sink topics.
c.recreate(source_str)
c.recreate(sink_str)
# 3. Start Streams on the built topology as a Python task.
stop = start_streams_task(built_tn)

#

# 4. Get the generator.
gen = OrderGenerator()
# 5. Set up the producer.
pr = c.producer(source_str)
# 6. Loop (10 times): Generate a message + produce it to Kafka.
for _ in range(10):
    message_dict = gen.generate()
    pr.produce(message_dict["value"], key=message_dict["key"])
# 7. Close the producer.
pr.close()

#

print("Streams started...")



We can now stop our Streams task...

In [ ]:
await stop()

...and then print out the sink topic:

In [ ]:
c.cat(sink_str)

That's it. Congratulations - that was your first Streams topology in action :-)

That's how stream processing looks like 2026. Natural.